In [38]:
from pathlib import Path
import ast
import json
import re
import math

import numpy as np
import pandas as pd
from datasets import load_dataset

In [39]:
ATTRACTION_FEATURES_PATH = Path(
    r"C:\Users\negia\trip_plan\database\attractions\attraction_features.csv"
)

BASELINE_SUBMISSION_PATH = Path(
    r"C:\Users\negia\trip_plan\submissions_baseline_100\validation_gpt-3.5-turbo-0125_two-stage_submission.jsonl"
)

DWELL_SUBMISSION_PATH = Path(
    r"C:\Users\negia\trip_plan\submissions_dwell_100\validation_gpt-3.5-turbo-0125_two-stage_submission.jsonl"
)

In [40]:
def safe_literal_eval(x):
    if x is None:
        return None
    if isinstance(x, (list, dict)):
        return x
    try:
        return ast.literal_eval(x)
    except Exception:
        return None


def normalize_text(x):
    if x is None or pd.isna(x):
        return ""
    x = str(x).lower().strip()
    x = re.sub(r"[^a-z0-9]+", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalize_city(x):
    return normalize_text(x)


def is_missing_value(x):
    if x is None:
        return True
    x = str(x).strip()
    return x == "" or x == "-"


def split_semicolon_items(x):
    """
    Splits fields like:
    'A, City; B, City;'
    """
    if is_missing_value(x):
        return []
    parts = [p.strip() for p in str(x).split(";")]
    return [p for p in parts if p and p != "-"]


def extract_name_city(item):
    """
    Extracts attraction name and city from:
    'SkyWheel Myrtle Beach, Myrtle Beach'
    """
    item = str(item).strip()
    if "," in item:
        name, city = item.rsplit(",", 1)
        return name.strip(), city.strip()
    return item.strip(), ""


def parse_duration_minutes(text):
    """
    Extract explicit duration from text.

    Handles:
    'Duration: 6 hours 47 mins'
    '1 hours 40 minutes'
    '2 hours 28 minutes'
    '47 minutes'
    """
    if is_missing_value(text):
        return 0.0

    s = str(text).lower()

    m = re.search(
        r"(\d+(?:\.\d+)?)\s*hours?\s*(\d+(?:\.\d+)?)?\s*(?:mins?|minutes?)?",
        s,
    )
    if m:
        hours = float(m.group(1))
        mins = float(m.group(2)) if m.group(2) else 0.0
        return hours * 60 + mins

    m = re.search(r"(\d+(?:\.\d+)?)\s*(?:mins?|minutes?)", s)
    if m:
        return float(m.group(1))

    return 0.0

In [41]:
# Transparent assumptions.
# Only counted if the meal field is not "-".
MEAL_MINUTES = {
    "breakfast": 30.0,
    "lunch": 45.0,
    "dinner": 60.0,
}


def meal_load_minutes(day):
    load = 0.0

    for meal, minutes in MEAL_MINUTES.items():
        if not is_missing_value(day.get(meal, "-")):
            load += minutes

    return load

In [42]:
attractions = pd.read_csv(ATTRACTION_FEATURES_PATH)

required_cols = ["Name", "City", "predicted_dwell_minutes"]
missing_cols = [c for c in required_cols if c not in attractions.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

attractions["name_norm"] = attractions["Name"].apply(normalize_text)
attractions["city_norm"] = attractions["City"].apply(normalize_city)

print("Attraction rows:", len(attractions))
print("Missing predicted dwell:", attractions["predicted_dwell_minutes"].isna().sum())

display(attractions[["Name", "City", "predicted_dwell_minutes"]].head(10))

Attraction rows: 5302
Missing predicted dwell: 0


,Name,City,predicted_dwell_minutes
0,Aberdeen Maritime Museum,Aberdeen,85.2
1,Duthie Park,Aberdeen,89.9
2,Seaton Park,Aberdeen,85.1
3,Provost Skene's House,Aberdeen,84.7
4,Aberdeen Science Centre,Aberdeen,84.6
5,Johnston Gardens,Aberdeen,88.4
6,The Gordon Highlanders Museum,Aberdeen,82.0
7,Cruickshank Botanic Garden,Aberdeen,84.6
8,Hazlehead Park,Aberdeen,86.7
9,The Tolbooth Museum,Aberdeen,84.2


In [43]:
# Exact name + city lookup
attr_lookup = (
    attractions
    .groupby(["name_norm", "city_norm"], as_index=False)["predicted_dwell_minutes"]
    .median()
)

# Fallback name-only lookup
attr_lookup_name_only = (
    attractions
    .groupby("name_norm", as_index=False)["predicted_dwell_minutes"]
    .median()
)

print("Unique name+city lookup rows:", len(attr_lookup))
print("Unique name-only lookup rows:", len(attr_lookup_name_only))

Unique name+city lookup rows: 5288
Unique name-only lookup rows: 5138


In [44]:
def lookup_attraction_dwell(name, city=""):
    """
    First tries name + city.
    Then falls back to name-only.
    Returns np.nan if no match.
    """
    name_norm = normalize_text(name)
    city_norm = normalize_city(city)

    if not name_norm:
        return np.nan

    if city_norm:
        match = attr_lookup[
            (attr_lookup["name_norm"] == name_norm)
            & (attr_lookup["city_norm"] == city_norm)
        ]

        if len(match) > 0:
            return float(match["predicted_dwell_minutes"].iloc[0])

    match = attr_lookup_name_only[
        attr_lookup_name_only["name_norm"] == name_norm
    ]

    if len(match) > 0:
        return float(match["predicted_dwell_minutes"].iloc[0])

    return np.nan


def attraction_load_minutes(day, default_missing_dwell=np.nan):
    """
    Computes total attraction dwell time for one day.
    """
    items = split_semicolon_items(day.get("attraction", "-"))

    total = 0.0
    count = 0
    matched = 0
    missing_items = []

    for item in items:
        name, city = extract_name_city(item)

        if is_missing_value(name):
            continue

        count += 1
        dwell = lookup_attraction_dwell(name, city)

        if pd.isna(dwell):
            missing_items.append(item)

            if not pd.isna(default_missing_dwell):
                total += float(default_missing_dwell)
        else:
            total += float(dwell)
            matched += 1

    return total, count, matched, missing_items

In [45]:
def compute_day_load(day, default_missing_dwell=np.nan):
    """
    day_load_minutes =
    attraction dwell time
    + meal time
    + explicit transport duration
    """
    attraction_minutes, attraction_count, matched_count, missing_items = (
        attraction_load_minutes(
            day,
            default_missing_dwell=default_missing_dwell,
        )
    )

    meal_minutes = meal_load_minutes(day)
    transport_minutes = parse_duration_minutes(day.get("transportation", "-"))

    day_load_minutes = attraction_minutes + meal_minutes + transport_minutes

    return {
        "day": day.get("days", day.get("day", None)),
        "current_city": day.get("current_city", ""),
        "attraction_minutes": attraction_minutes,
        "meal_minutes": meal_minutes,
        "transport_minutes": transport_minutes,
        "day_load_minutes": day_load_minutes,
        "attraction_count": attraction_count,
        "matched_attraction_count": matched_count,
        "missing_attraction_count": len(missing_items),
        "missing_attractions": missing_items,
    }

In [46]:
train = load_dataset("osunlp/TravelPlanner", "train")["train"]

print(train)
print(train.column_names)
print("Rows:", len(train))

Dataset({
    features: ['org', 'dest', 'days', 'visiting_city_number', 'date', 'people_number', 'local_constraint', 'budget', 'query', 'level', 'annotated_plan', 'reference_information'],
    num_rows: 45
})
['org', 'dest', 'days', 'visiting_city_number', 'date', 'people_number', 'local_constraint', 'budget', 'query', 'level', 'annotated_plan', 'reference_information']
Rows: 45


In [47]:
def extract_plan_from_annotated_plan(annotated_plan_raw):
    """
    Train annotated_plan structure usually looks like:

    [
        query_dict,
        [day1, day2, day3, ..., {}, {}, {}]
    ]

    This extracts only real day dictionaries and removes padding {} entries.
    """
    obj = safe_literal_eval(annotated_plan_raw)

    if obj is None:
        return None

    candidate = None

    # Case: already a list of day dictionaries
    if isinstance(obj, list) and obj and all(isinstance(x, dict) for x in obj):
        candidate = obj

    # Case: nested list
    if isinstance(obj, list):
        for part in obj:
            if isinstance(part, list) and part and all(isinstance(x, dict) for x in part):
                if any("attraction" in x or "current_city" in x for x in part):
                    candidate = part
                    break

    if candidate is None:
        return None

    # Remove empty padding dictionaries and malformed non-day entries
    cleaned = []
    for d in candidate:
        if not isinstance(d, dict):
            continue

        if not d:
            continue

        if "days" not in d and "day" not in d:
            continue

        # Keep only dictionaries that look like actual plan days
        if any(k in d for k in ["current_city", "transportation", "breakfast", "attraction", "lunch", "dinner", "accommodation"]):
            cleaned.append(d)

    return cleaned if cleaned else None


train_plans = []
failed = 0

for i, row in enumerate(train):
    plan = extract_plan_from_annotated_plan(row["annotated_plan"])

    if plan:
        train_plans.append({
            "idx": i,
            "query": row["query"],
            "level": row["level"],
            "days_expected": row["days"],
            "plan": plan,
        })
    else:
        failed += 1

print("Extracted train annotated plans:", len(train_plans))
print("Failed:", failed)

check_df = pd.DataFrame([
    {
        "idx": x["idx"],
        "level": x["level"],
        "days_expected": x["days_expected"],
        "actual_days": len(x["plan"]),
    }
    for x in train_plans
])

display(check_df.head(20))
print("Total extracted train days:", check_df["actual_days"].sum())

Extracted train annotated plans: 45
Failed: 0


,idx,level,days_expected,actual_days
0,0,easy,3,3
1,1,easy,3,3
2,2,easy,3,3
3,3,easy,3,3
4,4,easy,3,3
5,5,easy,5,5
6,6,easy,5,5
7,7,easy,5,5
8,8,easy,5,5
9,9,easy,5,5


Total extracted train days: 225


In [48]:
example = train_plans[0]

print("Query:")
print(example["query"])

print("\nPlan:")
for day in example["plan"]:
    print(day)

Query:
Please help me plan a trip from St. Petersburg to Rockford spanning 3 days from March 16th to March 18th, 2022. The travel should be planned for a single person with a budget of $1,700.

Plan:
{'days': 1, 'current_city': 'from St. Petersburg to Rockford', 'transportation': 'Flight Number: F3573659, from St. Petersburg to Rockford, Departure Time: 15:40, Arrival Time: 17:04', 'breakfast': '-', 'attraction': '-', 'lunch': '-', 'dinner': 'Coco Bambu, Rockford', 'accommodation': 'Pure luxury one bdrm + sofa bed on Central Park, Rockford'}
{'days': 2, 'current_city': 'Rockford', 'transportation': '-', 'breakfast': 'Dial A Cake, Rockford', 'attraction': 'Burpee Museum of Natural History, Rockford;Midway Village Museum, Rockford;Discovery Center Museum, Rockford;', 'lunch': 'Flying Mango, Rockford', 'dinner': 'Cafe Southall, Rockford', 'accommodation': 'Pure luxury one bdrm + sofa bed on Central Park, Rockford'}
{'days': 3, 'current_city': 'from Rockford to St. Petersburg', 'transporta

In [49]:
train_day_rows = []

for plan_obj in train_plans:
    for day in plan_obj["plan"]:
        row = compute_day_load(day, default_missing_dwell=np.nan)

        row["plan_idx"] = plan_obj["idx"]
        row["level"] = plan_obj["level"]
        row["query"] = plan_obj["query"]

        train_day_rows.append(row)

train_day_loads_no_fallback = pd.DataFrame(train_day_rows)

print("Train evaluated days:", len(train_day_loads_no_fallback))
print("Total attractions:", train_day_loads_no_fallback["attraction_count"].sum())
print("Matched attractions:", train_day_loads_no_fallback["matched_attraction_count"].sum())
print("Missing attractions:", train_day_loads_no_fallback["missing_attraction_count"].sum())

display(train_day_loads_no_fallback.head())

Train evaluated days: 225
Total attractions: 338
Matched attractions: 338
Missing attractions: 0


,day,current_city,attraction_minutes,meal_minutes,transport_minutes,day_load_minutes,attraction_count,matched_attraction_count,missing_attraction_count,missing_attractions,plan_idx,level,query
0,1,from St. Petersburg to Rockford,0.0,60.0,0.0,60.0,0,0,0,[],0,easy,Please help me plan a trip from St. Petersburg...
1,2,Rockford,270.0,135.0,0.0,405.0,3,3,0,[],0,easy,Please help me plan a trip from St. Petersburg...
2,3,from Rockford to St. Petersburg,185.5,135.0,0.0,320.5,2,2,0,[],0,easy,Please help me plan a trip from St. Petersburg...
3,1,from Kansas City to Pensacola,0.0,60.0,842.0,902.0,0,0,0,[],1,easy,Can you provide a travel plan for 1 person dep...
4,2,Pensacola,346.0,135.0,0.0,481.0,4,4,0,[],1,easy,Can you provide a travel plan for 1 person dep...


In [50]:
missing_rows = train_day_loads_no_fallback[
    train_day_loads_no_fallback["missing_attraction_count"] > 0
]

print("Days with missing attraction matches:", len(missing_rows))

if len(missing_rows) > 0:
    display(missing_rows[[
        "plan_idx",
        "level",
        "day",
        "current_city",
        "missing_attraction_count",
        "missing_attractions",
    ]].head(20))

Days with missing attraction matches: 0


In [51]:
fallback_dwell = float(attractions["predicted_dwell_minutes"].median())

print("Fallback dwell for unmatched attractions:", fallback_dwell)

train_day_rows = []

for plan_obj in train_plans:
    for day in plan_obj["plan"]:
        row = compute_day_load(
            day,
            default_missing_dwell=fallback_dwell,
        )

        row["plan_idx"] = plan_obj["idx"]
        row["level"] = plan_obj["level"]
        row["query"] = plan_obj["query"]

        train_day_rows.append(row)

train_day_loads = pd.DataFrame(train_day_rows)

display(train_day_loads.head())

Fallback dwell for unmatched attractions: 90.2


,day,current_city,attraction_minutes,meal_minutes,transport_minutes,day_load_minutes,attraction_count,matched_attraction_count,missing_attraction_count,missing_attractions,plan_idx,level,query
0,1,from St. Petersburg to Rockford,0.0,60.0,0.0,60.0,0,0,0,[],0,easy,Please help me plan a trip from St. Petersburg...
1,2,Rockford,270.0,135.0,0.0,405.0,3,3,0,[],0,easy,Please help me plan a trip from St. Petersburg...
2,3,from Rockford to St. Petersburg,185.5,135.0,0.0,320.5,2,2,0,[],0,easy,Please help me plan a trip from St. Petersburg...
3,1,from Kansas City to Pensacola,0.0,60.0,842.0,902.0,0,0,0,[],1,easy,Can you provide a travel plan for 1 person dep...
4,2,Pensacola,346.0,135.0,0.0,481.0,4,4,0,[],1,easy,Can you provide a travel plan for 1 person dep...


In [52]:
describe_cols = [
    "day_load_minutes",
    "attraction_minutes",
    "meal_minutes",
    "transport_minutes",
    "attraction_count",
    "missing_attraction_count",
]

display(
    train_day_loads[describe_cols].describe(
        percentiles=[0.25, 0.5, 0.75, 0.9, 0.95]
    )
)

,day_load_minutes,attraction_minutes,meal_minutes,transport_minutes,attraction_count,missing_attraction_count
count,225.000000,225.000000,225.000000,225.000000,225.000000,225.0
mean,477.266222,132.630667,104.266667,240.368889,1.502222,0.0
std,278.038782,101.758078,42.573676,359.090483,1.146129,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,310.300000,0.000000,60.000000,0.000000,0.000000,0.0
50%,399.300000,170.300000,135.000000,0.000000,2.000000,0.0
75%,588.000000,184.900000,135.000000,387.000000,2.000000,0.0
90%,868.600000,268.000000,135.000000,818.600000,3.000000,0.0
95%,1047.800000,278.180000,135.000000,1008.800000,3.000000,0.0
max,1539.000000,371.600000,135.000000,1341.000000,4.000000,0.0


In [53]:
train_day_loads["activity_load_minutes"] = (
    train_day_loads["attraction_minutes"] + train_day_loads["meal_minutes"]
)

train_day_loads["full_day_load_minutes"] = (
    train_day_loads["attraction_minutes"]
    + train_day_loads["meal_minutes"]
    + train_day_loads["transport_minutes"]
)

display(
    train_day_loads[
        [
            "activity_load_minutes",
            "full_day_load_minutes",
            "attraction_minutes",
            "meal_minutes",
            "transport_minutes",
            "attraction_count",
            "missing_attraction_count",
        ]
    ].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95])
)

,activity_load_minutes,full_day_load_minutes,attraction_minutes,meal_minutes,transport_minutes,attraction_count,missing_attraction_count
count,225.000000,225.000000,225.000000,225.000000,225.000000,225.000000,225.0
mean,236.897333,477.266222,132.630667,104.266667,240.368889,1.502222,0.0
std,135.713761,278.038782,101.758078,42.573676,359.090483,1.146129,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0
25%,75.000000,310.300000,0.000000,60.000000,0.000000,0.000000,0.0
50%,285.400000,399.300000,170.300000,135.000000,0.000000,2.000000,0.0
75%,318.200000,588.000000,184.900000,135.000000,387.000000,2.000000,0.0
90%,403.000000,868.600000,268.000000,135.000000,818.600000,3.000000,0.0
95%,413.180000,1047.800000,278.180000,135.000000,1008.800000,3.000000,0.0
max,506.600000,1539.000000,371.600000,135.000000,1341.000000,4.000000,0.0


In [54]:
activity_thresholds = {
    "activity_p75_train_human": float(train_day_loads["activity_load_minutes"].quantile(0.75)),
    "activity_p90_train_human": float(train_day_loads["activity_load_minutes"].quantile(0.90)),
    "activity_p95_train_human": float(train_day_loads["activity_load_minutes"].quantile(0.95)),
}

full_day_thresholds = {
    "full_day_p75_train_human": float(train_day_loads["full_day_load_minutes"].quantile(0.75)),
    "full_day_p90_train_human": float(train_day_loads["full_day_load_minutes"].quantile(0.90)),
    "full_day_p95_train_human": float(train_day_loads["full_day_load_minutes"].quantile(0.95)),
}

print("Activity-load thresholds:")
print(activity_thresholds)

print("\nFull-day-load thresholds:")
print(full_day_thresholds)

Activity-load thresholds:
{'activity_p75_train_human': 318.2, 'activity_p90_train_human': 403.0, 'activity_p95_train_human': 413.18}

Full-day-load thresholds:
{'full_day_p75_train_human': 588.0, 'full_day_p90_train_human': 868.6, 'full_day_p95_train_human': 1047.7999999999993}


In [55]:
display(
    train_day_loads
    .sort_values("day_load_minutes", ascending=False)
    .head(10)[[
        "plan_idx",
        "level",
        "day",
        "current_city",
        "day_load_minutes",
        "attraction_minutes",
        "meal_minutes",
        "transport_minutes",
        "attraction_count",
        "missing_attraction_count",
    ]]
)

,plan_idx,level,day,current_city,day_load_minutes,attraction_minutes,meal_minutes,transport_minutes,attraction_count,missing_attraction_count
46,10,easy,7,from Vernal(Utah) to Houston,1539.0,174.0,135.0,1230.0,2,0
40,10,easy,1,from Houston to Salt Lake City(Utah),1401.0,0.0,60.0,1341.0,0,0
87,19,medium,1,from Fort Lauderdale to Milwaukee,1326.0,0.0,60.0,1266.0,0,0
89,19,medium,3,from Milwaukee to Fort Lauderdale,1296.0,0.0,30.0,1266.0,0,0
74,14,easy,7,from Devils Lake(Wisconsin) to Punta Gorda,1291.0,0.0,30.0,1261.0,0,0
165,35,hard,1,from Atlanta to Bemidji(Minnesota),1251.0,0.0,60.0,1191.0,0,0
68,14,easy,1,from Punta Gorda to Marquette(Wisconsin),1244.0,0.0,60.0,1184.0,0,0
99,21,medium,5,from Baton Rouge(Louisiana) to New York,1233.0,0.0,30.0,1203.0,0,0
190,40,hard,1,from Indianapolis to Grand Junction(Colorado),1221.0,0.0,60.0,1161.0,0,0
115,25,medium,1,from Santa Ana to Abilene(Texas),1140.0,0.0,60.0,1080.0,0,0


In [56]:
def load_jsonl(path):
    rows = []

    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()

            if line:
                rows.append(json.loads(line))

    return rows

In [57]:
def evaluate_submission(submission_path, system_name, thresholds, default_missing_dwell):
    submissions = load_jsonl(submission_path)

    day_rows = []
    plan_rows = []

    for item in submissions:
        plan_idx = item.get("idx")
        plan = item.get("plan")

        if not plan:
            plan_rows.append({
                "system": system_name,
                "idx": plan_idx,
                "valid_plan": False,
                "num_days": 0,
            })
            continue

        plan_day_rows = []

        for day in plan:
            row = compute_day_load(
                day,
                default_missing_dwell=default_missing_dwell,
            )

            row["system"] = system_name
            row["idx"] = plan_idx

            plan_day_rows.append(row)
            day_rows.append(row)

        plan_summary = {
            "system": system_name,
            "idx": plan_idx,
            "valid_plan": True,
            "num_days": len(plan_day_rows),
        }

        for t_name, t_value in thresholds.items():
            day_passes = [
                r["day_load_minutes"] <= t_value
                for r in plan_day_rows
            ]

            plan_summary[f"plan_pass_{t_name}"] = all(day_passes)
            plan_summary[f"overloaded_days_{t_name}"] = int(
                sum(not x for x in day_passes)
            )

        plan_rows.append(plan_summary)

    day_df = pd.DataFrame(day_rows)
    plan_df = pd.DataFrame(plan_rows)

    if len(day_df) > 0:
        for t_name, t_value in thresholds.items():
            day_df[f"day_pass_{t_name}"] = (
                day_df["day_load_minutes"] <= t_value
            )

            day_df[f"overload_minutes_{t_name}"] = np.maximum(
                0,
                day_df["day_load_minutes"] - t_value,
            )

    return day_df, plan_df

In [58]:
def summarize_temporal(day_df, plan_df, thresholds):
    rows = []

    for t_name, t_value in thresholds.items():
        valid_plan_df = plan_df[plan_df["valid_plan"]]

        row = {
            "threshold": t_name,
            "threshold_minutes": t_value,
            "evaluated_plans": int(valid_plan_df.shape[0]),
            "evaluated_days": int(day_df.shape[0]),
            "day_temporal_feasibility_rate": float(day_df[f"day_pass_{t_name}"].mean()),
            "plan_temporal_feasibility_rate": float(valid_plan_df[f"plan_pass_{t_name}"].mean()),
            "overloaded_day_rate": float((~day_df[f"day_pass_{t_name}"]).mean()),
            "mean_day_load_minutes": float(day_df["day_load_minutes"].mean()),
            "median_day_load_minutes": float(day_df["day_load_minutes"].median()),
            "mean_overload_minutes": float(day_df[f"overload_minutes_{t_name}"].mean()),
            "mean_attraction_minutes_per_day": float(day_df["attraction_minutes"].mean()),
            "mean_meal_minutes_per_day": float(day_df["meal_minutes"].mean()),
            "mean_transport_minutes_per_day": float(day_df["transport_minutes"].mean()),
            "mean_attraction_count_per_day": float(day_df["attraction_count"].mean()),
            "total_missing_attractions": int(day_df["missing_attraction_count"].sum()),
        }

        rows.append(row)

    return pd.DataFrame(rows)

In [59]:
baseline_day_df, baseline_plan_df = evaluate_submission(
    BASELINE_SUBMISSION_PATH,
    system_name="baseline",
    thresholds=thresholds,
    default_missing_dwell=fallback_dwell,
)

dwell_day_df, dwell_plan_df = evaluate_submission(
    DWELL_SUBMISSION_PATH,
    system_name="dwell_aware",
    thresholds=thresholds,
    default_missing_dwell=fallback_dwell,
)

print("Baseline days:", len(baseline_day_df))
print("Dwell-aware days:", len(dwell_day_df))

print("Baseline plans:", len(baseline_plan_df))
print("Dwell-aware plans:", len(dwell_plan_df))

Baseline days: 432
Dwell-aware days: 370
Baseline plans: 100
Dwell-aware plans: 100


In [60]:
baseline_summary = summarize_temporal(
    baseline_day_df,
    baseline_plan_df,
    thresholds,
)

dwell_summary = summarize_temporal(
    dwell_day_df,
    dwell_plan_df,
    thresholds,
)

comparison = pd.concat(
    [
        baseline_summary.assign(system="baseline"),
        dwell_summary.assign(system="dwell_aware"),
    ],
    ignore_index=True,
)

display(comparison)

,threshold,threshold_minutes,evaluated_plans,evaluated_days,day_temporal_feasibility_rate,plan_temporal_feasibility_rate,overloaded_day_rate,mean_day_load_minutes,median_day_load_minutes,mean_overload_minutes,mean_attraction_minutes_per_day,mean_meal_minutes_per_day,mean_transport_minutes_per_day,mean_attraction_count_per_day,total_missing_attractions,system
0,p75_train_human,480.95,95,432,0.983796,0.947368,0.016204,205.831019,222.05,4.519792,104.356481,93.541667,7.93287,1.189815,118,baseline
1,p90_train_human,779.60,95,432,0.993056,0.968421,0.006944,205.831019,222.05,1.483102,104.356481,93.541667,7.93287,1.189815,118,baseline
2,p95_train_human,973.90,95,432,0.995370,0.978947,0.004630,205.831019,222.05,0.478472,104.356481,93.541667,7.93287,1.189815,118,baseline
3,fixed_480,480.00,95,432,0.983796,0.947368,0.016204,205.831019,222.05,4.535185,104.356481,93.541667,7.93287,1.189815,118,baseline
4,fixed_600,600.00,95,432,0.990741,0.968421,0.009259,205.831019,222.05,3.093981,104.356481,93.541667,7.93287,1.189815,118,baseline
5,p75_train_human,480.95,85,370,0.964865,0.870588,0.035135,223.255405,221.70,9.047973,98.282432,103.702703,21.27027,1.135135,18,dwell_aware
6,p90_train_human,779.60,85,370,0.991892,0.988235,0.008108,223.255405,221.70,3.880541,98.282432,103.702703,21.27027,1.135135,18,dwell_aware
7,p95_train_human,973.90,85,370,0.991892,0.988235,0.008108,223.255405,221.70,2.305135,98.282432,103.702703,21.27027,1.135135,18,dwell_aware
8,fixed_480,480.00,85,370,0.964865,0.870588,0.035135,223.255405,221.70,9.081351,98.282432,103.702703,21.27027,1.135135,18,dwell_aware
9,fixed_600,600.00,85,370,0.975676,0.917647,0.024324,223.255405,221.70,5.783514,98.282432,103.702703,21.27027,1.135135,18,dwell_aware


In [61]:
calibrated_comparison = comparison[
    comparison["threshold"].isin([
        "p75_train_human",
        "p90_train_human",
        "p95_train_human",
    ])
][[
    "system",
    "threshold",
    "threshold_minutes",
    "evaluated_plans",
    "evaluated_days",
    "day_temporal_feasibility_rate",
    "plan_temporal_feasibility_rate",
    "overloaded_day_rate",
    "mean_day_load_minutes",
    "mean_overload_minutes",
    "mean_attraction_count_per_day",
    "total_missing_attractions",
]]

display(calibrated_comparison)

,system,threshold,threshold_minutes,evaluated_plans,evaluated_days,day_temporal_feasibility_rate,plan_temporal_feasibility_rate,overloaded_day_rate,mean_day_load_minutes,mean_overload_minutes,mean_attraction_count_per_day,total_missing_attractions
0,baseline,p75_train_human,480.95,95,432,0.983796,0.947368,0.016204,205.831019,4.519792,1.189815,118
1,baseline,p90_train_human,779.60,95,432,0.993056,0.968421,0.006944,205.831019,1.483102,1.189815,118
2,baseline,p95_train_human,973.90,95,432,0.995370,0.978947,0.004630,205.831019,0.478472,1.189815,118
5,dwell_aware,p75_train_human,480.95,85,370,0.964865,0.870588,0.035135,223.255405,9.047973,1.135135,18
6,dwell_aware,p90_train_human,779.60,85,370,0.991892,0.988235,0.008108,223.255405,3.880541,1.135135,18
7,dwell_aware,p95_train_human,973.90,85,370,0.991892,0.988235,0.008108,223.255405,2.305135,1.135135,18


In [62]:
t_name = "p90_train_human"

all_days = pd.concat(
    [baseline_day_df, dwell_day_df],
    ignore_index=True,
)

all_days["overload_minutes"] = np.maximum(
    0,
    all_days["day_load_minutes"] - thresholds[t_name],
)

display(
    all_days
    .sort_values("overload_minutes", ascending=False)
    .head(20)[[
        "system",
        "idx",
        "day",
        "current_city",
        "day_load_minutes",
        "overload_minutes",
        "attraction_minutes",
        "meal_minutes",
        "transport_minutes",
        "attraction_count",
        "missing_attraction_count",
        "missing_attractions",
    ]]
)

,system,idx,day,current_city,day_load_minutes,overload_minutes,attraction_minutes,meal_minutes,transport_minutes,attraction_count,missing_attraction_count,missing_attractions
611,dwell_aware,50,8,from Dallas to San Antonio,1376.9,597.3,85.9,135.0,1156.0,1,0,[]
604,dwell_aware,50,1,from Columbus to Houston,1255.5,475.9,78.5,135.0,1042.0,1,0,[]
24,baseline,9,1,from Sarasota to Philadelphia,1165.3,385.7,87.3,105.0,973.0,1,0,[]
609,dwell_aware,50,6,from Houston to Dallas,1142.2,362.6,90.2,135.0,917.0,1,1,"[Dallas World Aquarium, Dallas]"
48,baseline,17,1,from St. Louis to Washington,989.2,209.6,90.2,135.0,764.0,1,1,"[National Mall, Washington]"
68,baseline,23,4,from Amarillo to Lubbock,825.0,45.4,183.0,60.0,582.0,2,0,[]
538,dwell_aware,32,2,Seattle,85.3,0.0,85.3,0.0,0.0,1,0,[]
530,dwell_aware,30,4,Indianapolis,0.0,0.0,0.0,0.0,0.0,0,0,[]
531,dwell_aware,30,5,Indianapolis,0.0,0.0,0.0,0.0,0.0,0,0,[]
532,dwell_aware,31,1,from Cedar Rapids to Denver,87.3,0.0,87.3,0.0,0.0,1,0,[]


In [63]:
parts_summary = (
    all_days
    .groupby("system")[[
        "day_load_minutes",
        "attraction_minutes",
        "meal_minutes",
        "transport_minutes",
        "attraction_count",
        "missing_attraction_count",
    ]]
    .mean()
    .reset_index()
)

display(parts_summary)

,system,day_load_minutes,attraction_minutes,meal_minutes,transport_minutes,attraction_count,missing_attraction_count
0,baseline,205.831019,104.356481,93.541667,7.93287,1.189815,0.273148
1,dwell_aware,223.255405,98.282432,103.702703,21.27027,1.135135,0.048649


In [64]:
OUTPUT_DIR = Path(r"C:\Users\negia\trip_plan\temporal_eval_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

train_day_loads.to_csv(
    OUTPUT_DIR / "train_human_day_loads.csv",
    index=False,
)

baseline_day_df.to_csv(
    OUTPUT_DIR / "baseline_temporal_day_level.csv",
    index=False,
)

dwell_day_df.to_csv(
    OUTPUT_DIR / "dwell_temporal_day_level.csv",
    index=False,
)

comparison.to_csv(
    OUTPUT_DIR / "temporal_feasibility_comparison.csv",
    index=False,
)

with open(OUTPUT_DIR / "temporal_thresholds.json", "w", encoding="utf-8") as f:
    json.dump(thresholds, f, indent=2)

print("Saved outputs to:", OUTPUT_DIR)

for p in OUTPUT_DIR.iterdir():
    print("-", p.name)

Saved outputs to: C:\Users\negia\trip_plan\temporal_eval_outputs
- baseline_temporal_day_level.csv
- dwell_temporal_day_level.csv
- temporal_feasibility_comparison.csv
- temporal_thresholds.json
- train_human_day_loads.csv


In [65]:
print("Baseline invalid plans:")
print(baseline_plan_df[~baseline_plan_df["valid_plan"]][["system", "idx", "valid_plan", "num_days"]])

print("Dwell invalid plans:")
print(dwell_plan_df[~dwell_plan_df["valid_plan"]][["system", "idx", "valid_plan", "num_days"]])

Baseline invalid plans:
      system  idx  valid_plan  num_days
20  baseline   21       False         0
47  baseline   48       False         0
58  baseline   59       False         0
83  baseline   84       False         0
89  baseline   90       False         0
Dwell invalid plans:
         system  idx  valid_plan  num_days
22  dwell_aware   23       False         0
25  dwell_aware   26       False         0
34  dwell_aware   35       False         0
35  dwell_aware   36       False         0
37  dwell_aware   38       False         0
39  dwell_aware   40       False         0
41  dwell_aware   42       False         0
45  dwell_aware   46       False         0
46  dwell_aware   47       False         0
50  dwell_aware   51       False         0
52  dwell_aware   53       False         0
53  dwell_aware   54       False         0
56  dwell_aware   57       False         0
59  dwell_aware   60       False         0
93  dwell_aware   94       False         0


In [67]:
# Add activity_load_minutes if the notebook did not create it
for df in [baseline_day_df, dwell_day_df]:
    if "activity_load_minutes" not in df.columns:
        df["activity_load_minutes"] = df["attraction_minutes"] + df["meal_minutes"]

t_name = "activity_p75_train_human"

all_days = pd.concat([baseline_day_df, dwell_day_df], ignore_index=True)

all_days["overload_minutes"] = np.maximum(
    0,
    all_days["activity_load_minutes"] - activity_thresholds[t_name]
)

display(
    all_days.sort_values("overload_minutes", ascending=False).head(20)[[
        "system",
        "idx",
        "day",
        "current_city",
        "activity_load_minutes",
        "overload_minutes",
        "attraction_minutes",
        "meal_minutes",
        "attraction_count",
        "missing_attraction_count",
        "missing_attractions",
    ]]
)

,system,idx,day,current_city,activity_load_minutes,overload_minutes,attraction_minutes,meal_minutes,attraction_count,missing_attraction_count,missing_attractions
788,dwell_aware,98,2,Orlando,654.3,336.1,519.3,135.0,6,1,"[Pure Bliss, Orlando]"
16,baseline,6,2,San Diego,566.1,247.9,431.1,135.0,5,0,[]
15,baseline,6,1,from Detroit to San Diego,491.1,172.9,431.1,60.0,5,0,[]
487,dwell_aware,19,2,San Francisco,455.5,137.3,320.5,135.0,4,0,[]
443,dwell_aware,4,3,Honolulu,455.0,136.8,320.0,135.0,4,0,[]
17,baseline,6,3,from San Diego to Detroit,403.8,85.6,328.8,75.0,4,0,[]
276,baseline,60,2,Houston,397.2,79.0,262.2,135.0,3,0,[]
289,baseline,63,2,Twin Falls,324.3,6.1,189.3,135.0,2,0,[]
484,dwell_aware,18,2,Palm Springs,321.3,3.1,186.3,135.0,2,0,[]
310,baseline,70,2,Belleville,321.2,3.0,186.2,135.0,2,1,"[Eckert's Belleville Farm, Belleville]"


In [69]:
invalid_dwell_idxs = [23, 26, 35, 36, 38, 40, 42, 46, 47, 51, 53, 54, 57, 60, 94]

for idx in invalid_dwell_idxs:
    path = Path(r"C:\Users\negia\trip_plan\outputs_dwell_100\validation") / f"generated_plan_{idx}.json"
    data = json.load(open(path, encoding="utf-8"))
    raw = data[-1].get("gpt-3.5-turbo-0125_two-stage_results")
    parsed = data[-1].get("gpt-3.5-turbo-0125_two-stage_parsed_results")

    print(idx, "| raw:", raw, "| parsed:", parsed)

23 | raw: Max Token Length Exceeded. | parsed: None
26 | raw: Max Token Length Exceeded. | parsed: None
35 | raw:  | parsed: None
36 | raw:  | parsed: None
38 | raw:  | parsed: None
40 | raw: Max Token Length Exceeded. | parsed: None
42 | raw: Max Token Length Exceeded. | parsed: None
46 | raw: Max Token Length Exceeded. | parsed: None
47 | raw: Max Token Length Exceeded. | parsed: None
51 | raw: Max Token Length Exceeded. | parsed: None
53 | raw: Max Token Length Exceeded. | parsed: None
54 | raw: Max Token Length Exceeded. | parsed: None
57 | raw: Max Token Length Exceeded. | parsed: None
60 | raw: Max Token Length Exceeded. | parsed: None
94 | raw: Max Token Length Exceeded. | parsed: None
